# tw-market-data-lake — Drive 故障注入測試（Phase B1）

**只寫入獨立測試資料夾** `MyDrive/00Quant_Research/_lake_fault_test/<時間>/`，不碰正式 `00Quant_Research/data`（程式會拒絕）。
不呼叫任何行情 API：pipeline 用離線假資料（tests/conftest.py），但走**真正的** store / manifest / lock 程式碼。

| 測試 | 判準 |
|---|---|
| T1 寫入協定比較（舊：copy2→staging→rename；新：直接寫最終檔名 + fsync） | 重新掛載後，新協定 0 個空檔/大小不符 |
| T2 pipeline（pilot + update）寫進測試根目錄 | 重新掛載後 `verify --all --deep` 全部 OK |
| T3 寫入途中 kill -9，換「新 runtime 視角」（重新掛載）後修復 | 修復後 verify OK；pointer 只指向完整版本 |
| T4 兩個 writer 同時跑 | 恰好一個被 lock 拒絕 |
| T5 stale lock | 預設拒絕；`--break-stale-lock` 才接手，舊 lock 移到 quarantine |

測試完成後，`_lake_fault_test` 資料夾可自行刪除（程式不刪任何東西）。

In [ ]:
# 1. 參數
N_FILES = 200
REPO_OWNER, REPO_NAME, BRANCH = "hh4832", "tw-market-data-lake", "main"
PROJECT_DIR = "/content/tw-market-data-lake"
import datetime, os
TS = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
TEST_ROOT = f"/content/drive/MyDrive/00Quant_Research/_lake_fault_test/{TS}"
print(TEST_ROOT)

In [ ]:
# 2. 掛載 Drive
from google.colab import drive
drive.mount("/content/drive")
def remount():
    drive.flush_and_unmount()
    drive.mount("/content/drive")
    print("[remounted] 之後讀到的是 Drive 真正存的內容")

In [ ]:
# 3. 取得程式（指定 branch）+ 依賴
import subprocess, sys
if os.path.exists(f"{PROJECT_DIR}/.git"):
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git", PROJECT_DIR], check=True)
print("HEAD:", subprocess.check_output(["git", "-C", PROJECT_DIR, "rev-parse", "HEAD"], text=True).strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml>=6", "tqdm>=4.66", "pytest>=8"], check=True)
ENV = dict(os.environ, PYTHONPATH=f"{PROJECT_DIR}/src", TW_LAKE_GIT_SHA="faulttest")
S = f"{PROJECT_DIR}/scripts/drive_fault_injection.py"
RESULTS = {}
def sh(*args, env=None):
    r = subprocess.run([sys.executable, *args], cwd=PROJECT_DIR, env=env or ENV, text=True, capture_output=True)
    print(r.stdout[-4000:], r.stderr[-2000:])
    return r.returncode

In [ ]:
# 4. T1 寫入協定比較：寫入 → 重新掛載 → 檢查
sh(S, "write-protocols", "--root", f"{TEST_ROOT}/T1", "--n", str(N_FILES), "--expected", "/content/fault_expected.json")
remount()
RESULTS["T1_protocols"] = sh(S, "check-protocols", "--expected", "/content/fault_expected.json")

In [ ]:
# 5. T2 pipeline（真正的 store/manifest 程式碼，假資料）
sh(S, "pipeline", "--data-root", f"{TEST_ROOT}/T2", "--mode", "pilot", "--end", "2024-03-15")
remount()
sh(S, "pipeline", "--data-root", f"{TEST_ROOT}/T2", "--mode", "update", "--end", "2024-03-29")
remount()
RESULTS["T2_pipeline_verify"] = sh("-m", "tw_lake.audit.verify", "--data-root", f"{TEST_ROOT}/T2", "--all", "--deep")

In [ ]:
# 6. T3 kill -9 寫入途中 → 重新掛載 → 修復 → 重新掛載 → 驗證
sh(S, "crash", "--data-root", f"{TEST_ROOT}/T3", "--after", "20")
remount()
sh(S, "pipeline", "--data-root", f"{TEST_ROOT}/T3", "--mode", "pilot", "--extra=--break-stale-lock",
   env=dict(ENV, TW_FAULT_STALE_MIN="0"))
remount()
RESULTS["T3_crash_recover_verify"] = sh("-m", "tw_lake.audit.verify", "--data-root", f"{TEST_ROOT}/T3", "--all", "--deep")

In [ ]:
# 7. T4 兩個 writer 同時跑；T5 stale lock
RESULTS["T4_two_writers"] = sh(S, "two-writers", "--data-root", f"{TEST_ROOT}/T4")
RESULTS["T5_stale_lock"] = sh(S, "stale-lock", "--data-root", f"{TEST_ROOT}/T5")
remount()

In [ ]:
# 8. 結果（0 = PASS）
for k, v in RESULTS.items():
    print(f"{k:28s} {'PASS' if v == 0 else 'FAIL'}")
print("\n全部 PASS 才建議合併 PR 並在正式資料跑 update（回補 83 個空檔）。請把這格輸出貼回給 Claude。")